In [ ]:
pip install pymupdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 57.9 MB/s eta 0:00:00


In [ ]:
import fitz

pdf = fitz.open("/content/drive/MyDrive/World_Checklist_of_Useful_Plant_Species_2020.pdf")

page = pdf[11]   # page 12

text = page.get_text()

print(text[:5000])

 World Checklist of Useful Plant Species (2020);***///!!!\\\|||###~~~@@@ Please contact Mauricio Diazgranados @ RBG Kew | 
m.diazgranados@kew.org if you have any comment. Database developed by RBG Kew -1234567890-abcdefghijklmnopqrstuvwxyz///\\\***  
Page 12 of 689 
 
Plantae  
Tracheophyta 
Cycadopsida 
Cycadales 
Cycadaceae 
 Cycas 
Cycas apoa K.D.Hill 
 
980285-1 | ME | [4, 5] 
Cycas circinalis L. 
 
328822-2 | HF ME PO SU | [1, 4, 5] 
Cycas inermis Lour. 
 
297004-1 | ME | [4] 
Cycas media R.Br. 
 
297013-1 | ME | [4] 
Cycas micronesica K.D.Hill 
 
980286-1 | HF ME | [4, 5] 
Cycas pectinata Buch.-Ham. 
 
297022-1 | HF ME | [4, 7] 
Cycas revoluta Thunb. 
 
328823-2 | EU GS HF MA ME | [1, 3, 4, 5, 7, 8] 
Cycas rumphii Miq. 
 
326820-2 | EU GS HF IF MA ME PO SU | [1, 3, 4, 5, 7, 8] 
Cycas siamensis Miq. 
 
297036-1 | EU HF ME | [3, 4, 7] 
Cycas taiwaniana Carruth. 
 
297043-1 | EU | [3] 
Cycas thouarsii R.Br. 
 
297044-1 | EU GS HF | [3, 8] 
Zamiaceae 
 Bowenia 
Bowenia serrulata (W.B

In [ ]:
import fitz
import pandas as pd
import re

CATEGORY_MAP = {
    "AF":"Animal Food",
    "EU":"Environmental Uses",
    "FU":"Fuels",
    "GS":"Gene Sources",
    "HF":"Human Food",
    "IF":"Invertebrate Food",
    "MA":"Materials",
    "ME":"Medicines",
    "PO":"Poisons",
    "SU":"Social Uses"
}

SOURCE_MAP = {
    "1":"Kew's Economic Botany Collection (EcBot)",
    "2":"European Red List of Medicinal Plants (Allen et al. 2014)",
    "3":"Germplasm Resources Information Network from USDA (GRIN)",
    "4":"Medicinal Plant Names Services (MPNS)",
    "5":"Useful Plants of New Guinea",
    "6":"Palms of the World Online",
    "7":"Plant Resources of South-East Asia (PROSEA)",
    "8":"Plant Resources for Tropical Africa (PROTA)",
    "9":"Survey of Economic Plants for Arid and Semi-Arid Lands (SEPASAL)",
    "10":"Useful Plant Project (UPP)",
    "11":"Useful Plants of West Tropical Africa (UPWTA)",
    "12":"Malaria and Fever in Latin America dataset",
    "13":"Crop Wild Relatives (CWR)"
}

pdf = fitz.open("/content/drive/MyDrive/World_Checklist_of_Useful_Plant_Species_2020.pdf")

rows=[]

# Skip obvious headers/taxonomy labels
skip_words = {
    "Plantae",
    "Tracheophyta",
    "Page"
}

for page_num in range(11,686):      # pages 12–686

    print(f"Page {page_num+1}")

    text = pdf[page_num].get_text()

    # remove page header garbage
    text = re.sub(
        r"World Checklist.*?Page \d+ of \d+",
        "",
        text,
        flags=re.DOTALL
    )

    lines = [
        x.strip()
        for x in text.split("\n")
        if x.strip()
    ]

    i=0

    while i < len(lines)-1:

        line=lines[i]

        # Skip taxonomy headings
        if (
            line in skip_words
            or len(line.split())==1
        ):
            i += 1
            continue

        # Species line:
        # starts with Genus species

        species_match = re.match(
            r"^([A-Z][a-z-]+)\s+([a-z-]+)",
            line
        )

        if species_match:

            species_author=line

            metadata=lines[i+1]

            # Metadata line should contain |
            if "|" in metadata:

                # ---------------------
                # LSID
                # ---------------------

                lsid_match=re.search(
                    r"^([\d\-]+)",
                    metadata
                )

                lsid=(
                    lsid_match.group(1)
                    if lsid_match
                    else ""
                )

                # ---------------------
                # Categories
                # ---------------------

                codes=re.findall(
                    r"\b(AF|EU|FU|GS|HF|IF|MA|ME|PO|SU)\b",
                    metadata
                )

                categories="; ".join(
                    CATEGORY_MAP[c]
                    for c in codes
                )

                # ---------------------
                # CWR
                # ---------------------

                cwr=bool(
                    re.search(
                        r"\bCWR\b",
                        metadata
                    )
                )

                # ---------------------
                # Sources
                # ---------------------

                source_match=re.search(
                    r"\[(.*?)\]",
                    metadata
                )

                source=""

                if source_match:

                    nums=re.findall(
                        r"\d+",
                        source_match.group(1)
                    )

                    source="; ".join(
                        SOURCE_MAP[n]
                        for n in nums
                        if n in SOURCE_MAP
                    )

                # ---------------------
                # Split species/author
                # ---------------------

                parts=species_author.split()

                species=" ".join(parts[:2])

                author=" ".join(parts[2:])

                rows.append({

                    "Species":species,
                    "Publication Author":author,
                    "LSID":lsid,
                    "Category of Use":categories,
                    "Crop Wild Relative":cwr,
                    "Source":source
                })

                i += 2
                continue

        i += 1

df = pd.DataFrame(rows)

print("\nRows extracted:",len(df))

df.to_csv(
    "plant_species_uses.csv",
    index=False
)

print("Saved plant_species_uses.csv")

Page 12
Page 13
Page 14
Page 15
Page 16
Page 17
Page 18
Page 19
Page 20
Page 21
Page 22
Page 23
Page 24
Page 25
Page 26
Page 27
Page 28
Page 29
Page 30
Page 31
Page 32
Page 33
Page 34
Page 35
Page 36
Page 37
Page 38
Page 39
Page 40
Page 41
Page 42
Page 43
Page 44
Page 45
Page 46
Page 47
Page 48
Page 49
Page 50
Page 51
Page 52
Page 53
Page 54
Page 55
Page 56
Page 57
Page 58
Page 59
Page 60
Page 61
Page 62
Page 63
Page 64
Page 65
Page 66
Page 67
Page 68
Page 69
Page 70
Page 71
Page 72
Page 73
Page 74
Page 75
Page 76
Page 77
Page 78
Page 79
Page 80
Page 81
Page 82
Page 83
Page 84
Page 85
Page 86
Page 87
Page 88
Page 89
Page 90
Page 91
Page 92
Page 93
Page 94
Page 95
Page 96
Page 97
Page 98
Page 99
Page 100
Page 101
Page 102
Page 103
Page 104
Page 105
Page 106
Page 107
Page 108
Page 109
Page 110
Page 111
Page 112
Page 113
Page 114
Page 115
Page 116
Page 117
Page 118
Page 119
Page 120
Page 121
Page 122
Page 123
Page 124
Page 125
Page 126
Page 127
Page 128
Page 129
Page 130
Page 131
Page 132

In [ ]:
import pandas as pd

# Load files
flowering = pd.read_csv("/content/drive/MyDrive/flowering_trees.csv")
uses = pd.read_csv("plant_species_uses.csv")

# ----------------------------
# Standardize species names
# ----------------------------

flowering["species_clean"] = (
    flowering["scientificname"]
    .astype(str)
    .str.strip()
    .str.lower()
)

uses["species_clean"] = (
    uses["Species"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ----------------------------
# Merge
# ----------------------------

merged = flowering.merge(
    uses[
        [
            "species_clean",
            "Category of Use",
            "Crop Wild Relative",
            "Source"
        ]
    ],
    on="species_clean",
    how="left"
)

# Remove helper column
merged.drop(
    columns=["species_clean"],
    inplace=True
)

# Save result
merged.to_csv(
    "flowering_trees_uses.csv",
    index=False
)

# Summary
matched = merged["Category of Use"].notna().sum()
unmatched = merged["Category of Use"].isna().sum()

print(f"Matched species: {matched}")
print(f"Unmatched species retained: {unmatched}")
print(f"Total rows: {len(merged)}")

print("\nSaved: flowering_trees_uses.csv")

Matched species: 12123
Unmatched species retained: 41355
Total rows: 53478

Saved: flowering_trees_uses.csv
